In [212]:
from sympy import symbols, Rational
from sympy.utilities.codegen import codegen
from sympy.codegen.rewriting import optimize, optims_c99
from sympy.simplify.cse_main import cse
import sympy as sp
from sympy import S
from sympy.printing.c import C99CodePrinter, Assignment
from sympy import init_printing
from codegen.codegen_utils import * 
init_printing()
printer = MyPrinter() 

In [213]:
def der_vec(vec,name):
    out = [] 
    n = vec.shape[0]
    for idir in range(3):
        _out = sp.Matrix([0]*n)
        for i in range(n):
            _out[i] = sp.symbols(f"d{name}_dx[{ i + n * idir}]")
        out.append(_out)
    return out

def second_der_vec(vec, name):
    n = vec.shape[0]
    out = [[sp.Matrix([0]*n) for _ in range(3)] for _ in range(3)]

    icomp = 0
    for idir in range(3):
        for jdir in range(idir, 3):  # upper triangle only
            for i in range(n):
                out[idir][jdir][i] = out[jdir][idir][i] = sp.symbols(f"dd{name}_dx2[{icomp}]")
                icomp += 1
    return out

def upwind_der_vec(vec,name):
    out = [] 
    n = vec.shape[0]
    for idir in range(3):
        _out = sp.Matrix([0]*n)
        for i in range(n):
            _out[i] = sp.symbols(f"d{name}_dx_upwind[{ i + n * idir}]")
        out.append(_out)
    return out

In [214]:
def der_symm_tens(mat,name):
    rows, cols = mat.shape
    if rows != cols: raise ValueError("Non-square tensor cannot be symmetric")
    n = rows 
    ncomps = n * (n+1)//2 
    out = [] 
    for idir in range(3):
        _out = sp.MutableDenseMatrix(n, n, [0]* (n*n))
        icomp = 0 
        for i in range(n):
            for j in range(i,n):
                # order idir, i, j
                _out[i,j] = _out[j,i] = sp.symbols(f"d{name}_dx[{icomp + ncomps * idir}]")
                icomp += 1
        out.append(_out)
    return out

def upwind_der_symm_tens(mat,name):
    rows, cols = mat.shape
    if rows != cols: raise ValueError("Non-square tensor cannot be symmetric")
    n = rows 
    ncomps = n * (n+1)//2 
    out = [] 
    for idir in range(3):
        _out = sp.MutableDenseMatrix(n, n, [0]* (n*n))
        icomp = 0 
        for i in range(n):
            for j in range(i,n):
                # order idir, i, j
                _out[i,j] = _out[j,i] = sp.symbols(f"d{name}_dx_upwind[{icomp + ncomps * idir}]")
                icomp += 1
        out.append(_out)
    return out

import sympy as sp

def second_der_symm_tens(mat, name):
    n = mat.shape[0]
    if n != mat.shape[1]:
        raise ValueError("Non-square tensor cannot be symmetric")
    ncomps = n * (n + 1) // 2  # unique components of tensor

    # Initialize 3x3 grid of n x n matrices
    out = [[sp.MutableDenseMatrix(n, n, [0]*(n*n)) for _ in range(3)] for _ in range(3)]

    icomp = 0
    for idir in range(3):
        for jdir in range(idir, 3):  # upper triangle in spatial indices
            _d = sp.MutableDenseMatrix(n, n, [0]*(n*n))
            for k in range(n):
                for l in range(k, n):  # upper triangle in tensor indices
                    symb = sp.symbols(f"dd{name}_dx2[{icomp}]")
                    _d[k, l] = _d[l, k] = symb  # tensor symmetry
                    icomp += 1
            out[idir][jdir] = _d
            out[jdir][idir] = _d  # enforce spatial symmetry
    return out  # out[idir][jdir][k,l] = ∂² A_{kl} / ∂x_idir ∂x_jdir


def der_scalar(s,name):
    _d = sp.Matrix([0]*3)
    for idir in range(3):
        _d[idir] = sp.symbols(f"d{name}_dx[{idir}]", real=True)
    return _d 

def upwind_der_scalar(s,name):
    _d = sp.Matrix([0]*3)
    for idir in range(3):
        _d[idir] = sp.symbols(f"d{name}_dx_upwind[{idir}]", real=True)
    return _d 

def second_der_scalar(s,name):
    out = [sp.MutableDenseMatrix([0]*3) for _ in range(3)] 
    icomp = 0 
    for idir in range(3):
        for jdir in range(idir,3):
            out[idir][jdir] = out[jdir][idir] = sp.symbols(f"dd{name}_dx2[{icomp}]", real=True)
            icomp += 1 
    return out

In [215]:
# variable block
gtxx,gtxy,gtxz,gtyy,gtyz,gtzz = symbols("gtdd[0] gtdd[1] gtdd[2] gtdd[3] gtdd[4] gtdd[5]") 
gtdd = sp.Matrix([
    [gtxx,gtxy,gtxz],
    [gtxy,gtyy,gtyz],
    [gtxz,gtyz,gtzz]
])

gtXX = gtyy * gtzz - gtyz**2 
gtYY = gtxx * gtzz - gtxz**2 
gtZZ = gtxx * gtyy - gtxy**2 
gtXY = gtxz * gtyz - gtxy * gtzz 
gtXZ = gtxy * gtyz - gtxz * gtyy 
gtYZ = gtxy * gtxz - gtxx * gtyz 

gtuu = sp.Matrix([
    [gtXX,gtXY,gtXZ],
    [gtXY,gtYY,gtYZ],
    [gtXZ,gtYZ,gtZZ]
])

print(sp.simplify(gtuu * gtdd / sp.det(gtdd)))

Atxx,Atxy,Atxz,Atyy,Atyz,Atzz = symbols("Atdd[0] Atdd[1] Atdd[2] Atdd[3] Atdd[4] Atdd[5]") 
Atdd = sp.Matrix([
    [Atxx,Atxy,Atxz],
    [Atxy,Atyy,Atyz],
    [Atxz,Atyz,Atzz]
])
betaX, betaY, betaZ = symbols("betau[0] betau[1] betau[2]") 
betau = sp.Matrix([betaX,betaY,betaZ])

GammatX, GammatY, GammatZ = symbols("Gammatu[0] Gammatu[1] Gammatu[2]") 
Gammatu = sp.Matrix([GammatX, GammatY, GammatZ])

alpha, chi = symbols("alp chi")
theta, Khat = symbols("theta Khat", real=True)

# Hydro 
Strace, rho = symbols("S rho", real=True)
Sxx, Sxy, Sxz, Syy, Syz, Szz = symbols("Sij[0] Sij[1] Sij[2] Sij[3] Sij[4] Sij[5]", real=True)
Sdd = sp.Matrix(
    [
        [Sxx,Sxy,Sxz],
        [Sxy,Syy,Syz],
        [Sxz,Syz,Szz]
    ]
)
Sx, Sy, Sz = symbols("Si[0] Si[1] Si[2]", real=True)
Sd = sp.Matrix([Sx,Sy,Sz])

# Definition of K with Khat and theta 
K = Khat + S(2) * theta 

# Parameters 
kappa1, kappa2 = symbols("kappa1 kappa2", real=True)
muL,muS,eta = symbols("muL muS eta", real=True)

# Aij up up 
Atuu = gtuu * Atdd * gtuu 

# Non conformal metric
gdd = gtdd/chi 

Matrix([[1, 0, 0], [0, 1, 0], [0, 0, 1]])


In [216]:
dgtdd_dx = der_symm_tens(gtdd,"gtdd")
d2gtdd_dx2 = second_der_symm_tens(gtdd,"gtdd")
dgtdd_dx_upw = upwind_der_symm_tens(gtdd,"gtdd")

dAtdd_dx_upw = upwind_der_symm_tens(Atdd,"Atdd")
dAtdd_dx = der_symm_tens(Atdd,"Atdd")

dbetau_dx = der_vec(betau,"betau")
d2betau_dx2 = second_der_vec(betau,"betau")
dbetau_dx_upw = upwind_der_vec(betau,"betau")

dGammatu_dx = der_vec(Gammatu,"Gammatu")
dGammatu_dx_upw = upwind_der_vec(Gammatu,"Gammatu")

dalpha_dx = der_scalar(alpha,"alp")
d2alpha_dx2 = second_der_scalar(alpha,"alp")
dalpha_dx_upw = upwind_der_scalar(alpha,"alp")


dKhat_dx = der_scalar(Khat,"Khat")
dKhat_dx_upw = upwind_der_scalar(Khat,"Khat")


dchi_dx = der_scalar(chi, "chi")
d2chi_dx2 = second_der_scalar(chi,"chi")
dchi_dx_upw = upwind_der_scalar(chi,"chi")

dtheta_dx = der_scalar(theta, "theta")
dtheta_dx_upw = upwind_der_scalar(theta, "theta")


In [217]:
Gammatudd = []

for i in range(3):
    _gammaloc = sp.zeros(3,3)
    for j in range(3):
        for k in range(j,3):
            for l in range(3):
                _gammaloc[j,k] += 0.5 * gtuu[i,l] * ( -dgtdd_dx[l][j,k] + dgtdd_dx[j][l,k] + dgtdd_dx[k][l,j] )
            _gammaloc[k,j] = _gammaloc[j,k]
    Gammatudd.append(_gammaloc)
    

In [218]:
# Define GammatD^i = gtuu[j,k] Gammatudd^i_{jk}
GammaDu = sp.zeros(3,1)
for i in range(3):
    for j in range(3):
        for k in range(3):
            GammaDu[i] += sp.simplify(gtuu[j,k] * Gammatudd[i][j,k])


Throughout this notebook:
$$
\gamma_{ij} = \chi^{-1} \tilde{\gamma}_{ij}
$$
So that
$$
\sqrt{\gamma} = \chi^{-3/2}
$$

Evolution of chi:
$$
\partial_t \chi = \frac{2}{3} \chi \left[ \alpha (\hat{K} + 2\Theta) - D_i \beta^i \right]
$$
Here we rewrite the covariant derivative of the shift as:
$$
D_i \beta^i = \frac{1}{\sqrt{\gamma}} \partial_i ( \sqrt{\gamma} \beta^i )
$$
Since $\chi = \gamma^{-1/3}$ $\sqrt{\gamma} = \chi^{-3/2}$
$$
D_i \beta^i = \chi^{3/2} \partial_i (\chi^{-3/2} \beta^i) = \partial_i \beta^i - \frac{3}{2} \chi^{-1} \beta^i \partial_i \chi 
$$
So all in all 
$$
\partial_t \chi = \frac{2}{3} \chi \left[ \alpha (\hat{K} + 2\Theta) - \partial_i \beta^i \right] + \beta^i \partial_i \chi 
$$

In [219]:
dchi_dt = Rational(2,3) * chi * alpha * (Khat+S(2)*theta)  
for i in range(3):
    dchi_dt += betau[i] * dchi_dx_upw[i] - Rational(2,3) * chi * dbetau_dx[i][i]

Evolution of gammatilde:
$$
\partial_t \tilde{\gamma}_{ij} = - 2 \alpha \tilde{A}_{ij} + \beta^k \partial_k \tilde{\gamma}_{ij} + 2 \tilde{\gamma}_{k(i}\partial_{j)}\beta^k - \frac{2}{3} \tilde{\gamma}_{ij} \partial_k \beta^k
$$

In [220]:
dgtdd_dt = - 2 * alpha * Atdd 
for i in range(3):
    dgtdd_dt += betau[i] * dgtdd_dx_upw[i] - Rational(2,3) * gtdd * dbetau_dx[i][i]

for i in range(3):
    for j in range(3):
        for k in range(3):
            dgtdd_dt[i,j] += gtdd[k,i] * dbetau_dx[j][k] + gtdd[k,j] * dbetau_dx[i][k]


Evolution of $\hat{K}$:

We need the term:
$$
D^i D_i \alpha = \Psi^{-6} \tilde{D}_i \Psi^{6} D^i \alpha = \Psi^{-4} \left( \tilde{D}_i \tilde{D}^i \alpha + 2 \tilde{D}_i \log{\Psi} \tilde{D}^i \alpha \right)
$$
Which is just 
$$
D^i D_i \alpha = \chi \left( \tilde{D}_i \tilde{D}^i \alpha - \frac{1}{2} \tilde{D}_i \log{\chi} \tilde{D}^i \alpha \right)
$$
which expands to 
$$
D^i D_i \alpha = \chi \left( \tilde{\gamma}^{ij} \partial_i \partial_j \alpha - (\tilde{\Gamma}_d)^i \partial_i \alpha - \frac{1}{2} \tilde{D}_i \log{\chi} \tilde{D}^i \alpha \right)
$$

In [221]:
DiDialp = - chi * GammaDu.dot(dalpha_dx)
for i in range(3):
    for j in range(3):
        DiDialp += chi*gtuu[i,j] * d2alpha_dx2[i][j]
DiDialp -= Rational(1,2) * sp.trace(gtuu * (dchi_dx*dalpha_dx.T))
DiDialp = sp.simplify(DiDialp)

Full equation reads:
$$
\partial_t \hat{K} = - D^i D_i \alpha + \alpha \left[ \tilde{A}_{ij} \tilde{A}^{ij} + \frac{1}{3} \left( \hat{K} + 2\Theta \right)^2 \right] + 4 \pi \alpha \left( S + \rho \right) + \alpha \, \kappa_1 \,(1-\kappa_2) \, \Theta + \beta^i \partial_i \hat{K}

In [222]:
dKhat_dt = -DiDialp 
dKhat_dt += alpha*(sp.trace(Atuu*Atdd) + Rational(1,3)*(Khat+S(2)*theta)**2)
dKhat_dt += S(4) * sp.pi * alpha * (Strace+rho) 
dKhat_dt += alpha * kappa1*(S(1)-kappa2)*theta 
dKhat_dt += betau.dot(dKhat_dx_upw)

Evolution of $\tilde{\Gamma}^i$
$$
\partial_t \tilde{\Gamma}^i = -2 \tilde{A}^{ij}\partial_j \alpha + 2 \alpha \left[ \tilde{\Gamma}^i_{jk} \tilde{A}^{jk} -\frac{3}{2} \tilde{A}^{ij} \partial_j \log{\chi} - \frac{1}{3} \tilde{\gamma}^{ij} \partial_j \left( 2 \hat{K} + \Theta \right) - 8\pi \tilde{\gamma}^{ij} S_j \right] \\  + \tilde{\gamma}^{jk} \partial_j \partial_k \beta^i + \frac{1}{3} \tilde{\gamma}^{ij} \partial_j \partial_k \beta^k + \beta^j \partial_j \tilde{\Gamma}^i - (\tilde{\Gamma}_d)^j \partial_j \beta^i + \frac{2}{3} (\tilde{\Gamma}_d)^i \partial_j \beta^j - 2 \alpha \, \kappa_1 \left[ \tilde{\Gamma}^i - (\tilde{\Gamma}_d)^i \right]
$$ 

In [223]:
test = - S(2) * Atuu * dalpha_dx
test += S(2) * alpha * sp.Matrix([sp.trace(_gL*Atuu) for _gL in Gammatudd])
test -= S(3) * alpha * Atuu * dchi_dx/chi 
test -= Rational(2,3) * alpha * gtuu * (S(2)*dKhat_dx + dtheta_dx)
test -= S(16) * alpha * sp.pi * gtuu * Sd 
for j in range(3): 
    for k in range(3):
        test += gtuu[j,k] * d2betau_dx2[j][k]
        for i in range(3):
            test[i] += Rational(1,3) * gtuu[i,j] * d2betau_dx2[j][k][k]
for j in range(3):
    test += betau[j] * dGammatu_dx_upw[j] - GammaDu[j] * dbetau_dx[j]
    test += Rational(2,3) * GammaDu * dbetau_dx[j][j]

test -= S(2) * alpha * kappa1 * (Gammatu - GammaDu)


In [224]:
dGammatu_dt = - 2 * alpha * kappa1 * (Gammatu - GammaDu)
for i in range(3):
    for j in range(3):
        dGammatu_dt[i] -= S(2) * Atuu[i,j] * dalpha_dx[j]  
        dGammatu_dt[i] -= S(3) * alpha * Atuu[i,j] * dchi_dx[j]/chi
        dGammatu_dt[i] -= Rational(2,3) * alpha * gtuu[i,j] * ( S(2) * dKhat_dx[j] + dtheta_dx[j] )
        dGammatu_dt[i] -= S(16) * alpha * sp.pi * gtuu[i,j] * Sd[j]  
        dGammatu_dt[i] += betau[j] * dGammatu_dx_upw[j][i] - GammaDu[j] * dbetau_dx[j][i]
        dGammatu_dt[i] += Rational(2,3) * GammaDu[i] * dbetau_dx[j][j]
        for k in range(3):
            dGammatu_dt[i] += S(2) * alpha * Gammatudd[i][j,k] * Atuu[j,k]
            dGammatu_dt[i] += gtuu[j,k] * d2betau_dx2[j][k][i]
            dGammatu_dt[i] += Rational(1,3) * gtuu[i,j] * d2betau_dx2[j][k][k]

In [225]:
sp.simplify(dGammatu_dt-test)

Ricci tensor:
- Part 1: $\tilde{R}_{ij}$
- Part 2: Trace 
- Part 3: $R^{\chi}_{ij}$

First we compute
$$
\tilde{\Gamma}_{ijk} = \tilde{\gamma}_{il} \tilde{\Gamma}^l_{jk}
$$

In [226]:
Gammatddd = [] # Gamma_{ijk}
for i in range(3):
    _gammaloc = sp.zeros(3,3)
    for j in range(3):
        for k in range(j,3):
            for l in range(3):
                _gammaloc[j,k] += gtdd[i,l] * Gammatudd[l][j,k]
            _gammaloc[k,j] = _gammaloc[j,k]
    Gammatddd.append(_gammaloc)

Now:
$$
\tilde{R}_{ij} = -\frac{1}{2} \tilde{\gamma}^{lm} \partial_l \partial_m \tilde{\gamma}_{ij} + \tilde{\gamma}_{k (i} \partial_{j)} \tilde{\Gamma}^k + (\tilde{\Gamma}_d)^k \tilde{\Gamma}_{(ij)k} + \tilde{\gamma}^{lm} \left( 2 {\tilde{\Gamma}^k}_{l(i} \tilde{\Gamma}_{j) km}  + {\tilde{\Gamma}^k}_{im} \tilde{\Gamma}_{klj} \right)
$$

In [227]:
test = sp.zeros(3,3)
for l in range(3):
    for m in range(3):
        test -= Rational(1,2) * gtuu[l,m] * d2gtdd_dx2[l][m]
for i in range(3):
    for j in range(3):
        for k in range(3):
            test[i,j] += Rational(1,2) * (gtdd[k,i] * dGammatu_dx[j][k] + gtdd[k,j] * dGammatu_dx[i][k]) 
            test[i,j] += Rational(1,2) * GammaDu[k] * (Gammatddd[i][j,k] + Gammatddd[j][i,k])

for i in range(3):
    for j in range(3):
        for k in range(3):
            for l in range(3):
                for m in range(3):
                    test[i,j] += gtuu[l,m] * (Gammatudd[k][l,i] * Gammatddd[j][k,m]+Gammatudd[k][l,j] * Gammatddd[i][k,m] + Gammatudd[k][i,m] * Gammatddd[k][l,j])

In [228]:
Rtdd = sp.zeros(3,3)
for i in range(3):
    for j in range(i,3):
        for k in range(3):
            Rtdd[i,j] += Rational(1,2) * (gtdd[k,i] * dGammatu_dx[j][k] + gtdd[k,j] * dGammatu_dx[i][k])  + Rational(1,2) * GammaDu[k] * (Gammatddd[i][j,k] + Gammatddd[j][i,k])
            for l in range(3):
                Rtdd[i,j] -= Rational(1,2) * gtuu[k,l] * d2gtdd_dx2[k][l][i,j] 
                for m in range(3):
                    Rtdd[i,j] += gtuu[l,m] * (Gammatudd[k][l,i] * Gammatddd[j][k,m] + Gammatudd[k][l,j] * Gammatddd[i][k,m] + Gammatudd[k][i,m] * Gammatddd[k][l,j])
        Rtdd[j,i] = Rtdd[i,j]
Rtdd = sp.MutableDenseMatrix(sp.simplify(Rtdd))
print(Rtdd.is_symmetric())

True


In [229]:
print(sp.simplify(test-Rtdd))

Matrix([[0, 0, 0], [0, 0, 0], [0, 0, 0]])


Next: 
$$
R = \Psi^{-4} \tilde{R} - 8 \Psi^{-5} \tilde{D}_i \tilde{D}^i \Psi =  \chi \tilde{R} + \left\{ 2\tilde{\gamma}^{ij} \partial_i \partial_j \chi - 2 (\tilde{\Gamma}_d)^i \partial_i \chi - \frac{5}{2 \chi} \tilde{\gamma}^{ij} \partial_i \chi \partial_j \chi  \right\}
$$
The second term comes from here: 
$$
-8 \Psi^{-5} \tilde{D}_i \tilde{D}^i \Psi = -8 \Psi^{-5} \left\{ \tilde{\gamma}^{ij} \partial_i \partial_j \Psi - \tilde{\Gamma_d}^i \partial_i \Psi \right\}
$$
Now using $\Psi = \chi^{-1/4}$ we get 
$$
-8 \chi^{5/4} \left\{ -\frac{1}{4} \tilde{\gamma}^{ij} \partial_i ( \chi^{-5/4} \partial_j \chi ) + \frac{1}{4}  \chi^{-5/4}  \tilde{\Gamma_d}^i \partial_i \chi \right\}
$$
which becomes 
$$
-8 \chi^{5/4} \left\{ -\frac{1}{4} \chi^{-5/4} \tilde{\gamma}^{ij} \partial_i  \partial_j \chi + \frac{5}{16} \chi^{-9/4} (\partial_i \chi) (\partial_j \chi) + \frac{1}{4}  \chi^{-5/4}  \tilde{\Gamma_d}^i \partial_i \chi \right\}
$$
and finally we get the result 
$$
2 \tilde{\gamma}^{ij} \partial_i  \partial_j \chi - 2 \tilde{\Gamma_d}^i \partial_i \chi - \frac{5}{2 \chi} (\partial_i \chi) (\partial_j \chi)
$$

In [ ]:
# Trace of Ricci of conf metric 
Rttrace = sp.simplify(sp.trace(gtuu*Rtdd))

# Trace of conf ricci
test = - S(2) * GammaDu.dot(dchi_dx)
test -= Rational(5,2) * sp.trace(gtuu*dchi_dx*dchi_dx.T) / chi  
for i in range(3):
    for j in range(3):
        test += S(2) * gtuu[i,j] * d2chi_dx2[i][j]

Rchitrace = 0 
for i in range(3):
    Rchitrace -= S(2) * GammaDu[i] * dchi_dx[i] 
    for j in range(3):
        Rchitrace +=  gtuu[i,j] * ( S(2)*d2chi_dx2[i][j] - Rational(5,2) * dchi_dx[i] * dchi_dx[j] / chi ) 
print(sp.simplify(test-Rchitrace))

# Trace of the full ricci 
Rtrace = chi * Rttrace + sp.simplify(Rchitrace)

0


$$
R^\chi_{ij} = \frac{1}{2\chi} \tilde{D}_i \tilde{D}_j \chi + \frac{1}{2\chi} \tilde{D}^l \tilde{D}_l \chi \tilde{\gamma}_{ij} - \frac{1}{4 \chi^2} \tilde{D}_i \chi \tilde{D}_j \chi - \frac{3}{4\chi^2}\tilde{\gamma}_{ij} \tilde{D}^l
\chi \tilde{D}_l \chi 
$$
Here
$$
\tilde{D}_l \tilde{D}^l \chi = \tilde{\gamma}^{i j} \partial_i \partial_j \chi - (\tilde{\Gamma}_d)^k \partial_k \chi
$$

In [231]:
# Dt_i Dt_j chi 
Dtijchi = sp.zeros(3,3)
for i in range(3):
    for j in range(i,3):
        Dtijchi[i,j] = d2chi_dx2[i][j]
        for k in range(3):
            Dtijchi[i,j] -= Gammatudd[k][i,j] * dchi_dx[k]
        Dtijchi[j,i] = Dtijchi[i,j]
test = sp.zeros(3,3)
for k in range(3):
    test -= dchi_dx[k] * Gammatudd[k]
for i in range(3):
    for j in range(3):
        test[i,j] += d2chi_dx2[i][j]
print(sp.simplify(test-Dtijchi))
# Dt_i Dt^i chi 
Dtiichi = sp.simplify(sp.trace(gtuu*Dtijchi))
# Dt_i chi Dt^i chi == gammat^{ij} d_i chi d_j chi 
DtichiDtichi = sp.simplify(sp.trace(gtuu*dchi_dx*dchi_dx.T))

Matrix([[0, 0, 0], [0, 0, 0], [0, 0, 0]])


In [232]:
Rchidd = Rational(1,2)/chi * Dtijchi - Rational(1,4)/chi**2 * dchi_dx * dchi_dx.T + (Rational(1,2)/chi * Dtiichi - Rational(3,4)/chi**2 * DtichiDtichi) * gtdd 
print(Rchidd.is_symmetric())
Rdd = Rchidd + Rtdd 
print(Rdd.is_symmetric())

# Just testing: compute explicitly the trace 
#Rtrace = 0 
#for i in range(3):
#    for j in range(3):
#        Rtrace += sp.simplify(chi * gtuu[i,j]*Rdd[i,j])
#Rtrace = sp.simplify(Rtrace)


True
True


Evolution of $\Theta$:
$$
\partial_t \Theta = \frac{1}{2} \alpha \left[ R - \tilde{A}_{ij} \tilde{A}^{ij} + \frac{2}{3} \left( \hat{K} + 2 \Theta \right)^2 \right] - \alpha \left[ 8\pi \rho + \kappa_1 (2 + \kappa_2) \Theta \right] + \beta^i \partial_i \Theta 
$$

In [233]:
dtheta_dt = Rational(1,2) * alpha * (Rtrace - sp.trace(Atuu*Atdd) + Rational(2,3)*(Khat+S(2)*theta)**2) - alpha * (S(8)*sp.pi*rho + kappa1*(S(2)+kappa2)*theta) + betau.dot(dtheta_dx_upw)
#dtheta_dt = sp.simplify(dtheta_dt)

Evolution of $ \tilde{A}_{ij} $

We also need the trace which we have from the $\hat{K}$ rhs.

In [234]:
# Dt_i Dt_j alpha 
Dtijalp = sp.zeros(3,3)
for i in range(3):
    for j in range(i,3):
        Dtijalp[i,j] = d2alpha_dx2[i][j]
        for k in range(3):
            Dtijalp[i,j] -= Gammatudd[k][i,j] * dalpha_dx[k]
        Dtijalp[j,i] = Dtijalp[i,j]
        
DiDjalp = Dtijalp + Rational(1,2) * ( dchi_dx * dalpha_dx.T + dalpha_dx * dchi_dx.T ) / chi - Rational(1,2) * sp.trace(gtuu * dchi_dx * dalpha_dx.T/chi)*gtdd 
print(DiDjalp.is_symmetric())

True


In [235]:
Atud = gtuu * Atdd 

First piece of the right hand side
$$
\Phi_{ij} := \left[ -D_i D_j \alpha + \alpha (R_{ij} - 8 \pi S_{ij} )\right]^{\rm TF}
$$
Note that our expressions for the ricci trace and for the trace of $D_i D_j \alpha$ already contain a factor of $\chi$. 

In [236]:
Phi = (- DiDjalp + alpha * (Rdd - S(8)*sp.pi*Sdd)) - Rational(1,3) * (-DiDialp + alpha * (Rtrace - S(8)*sp.pi * Strace)) * gdd 
print(Phi.is_symmetric())

True


Assemble the whole RHS:
$$ 
\partial_t \tilde{A}_{ij} = \chi \Phi_{ij} + \alpha \left[ K \tilde{A}_{ij} - 2 {\tilde{A}^k}_i \tilde{A}_{kj} \right] + \beta^k \partial_k \tilde{A}_{ij} + 2 \tilde{A}_{k(i}\partial_{j)}\beta^k -\frac{2}{3} \tilde{A}_{ij} \partial_k \beta^k
$$

In [237]:
dAtdd_dt = alpha * (Khat+S(2)*theta) * Atdd 
for i in range(3):
    for j in range(i,3):
        for k in range(3):
            dAtdd_dt[i,j] -= S(2) * alpha * Atud[k,i] * Atdd[k,j] 
            dAtdd_dt[i,j] += Atdd[k,i] * dbetau_dx[j][k] + Atdd[k,j] * dbetau_dx[i][k]
            dAtdd_dt[i,j] -= Rational(2,3) * Atdd[i,j] * dbetau_dx[k][k]
            dAtdd_dt[i,j] += betau[k] * dAtdd_dx_upw[k][i,j]
        dAtdd_dt[j,i] = dAtdd_dt[i,j]
dAtdd_dt += chi * Phi 

test = chi * Phi + alpha * ((Khat+S(2)*theta)*Atdd - S(2) * Atud.T * Atdd ) 
for k in range(3):
    test += betau[k] * dAtdd_dx_upw[k]
for i in range(3):
    for j in range(3):
        for k in range(3):
            test[i,j] += Atdd[k,i] * dbetau_dx[j][k] + Atdd[k,j] * dbetau_dx[i][k]
for k in range(3):
    test -= Rational(2,3) * dbetau_dx[k][k] * Atdd 
print(sp.simplify(dAtdd_dt-test))

Matrix([[0, 0, 0], [0, 0, 0], [0, 0, 0]])


Gauge evolution:

Lapse
$$
\partial_t \alpha = -\alpha^2 \mu_{\rm L} \hat{K} + \beta^i \partial_i \alpha 
$$
We choose $1+\log$ slicing: 
$$
\partial_t \alpha = -2 \alpha \hat{K} + \beta^i \partial_i \alpha 
$$
Shift 
$$
\partial_t \beta^i = \alpha^2 \mu_{\rm S} \tilde{\Gamma}^i - \eta \beta^i + \beta^j \partial_j \beta^i 
$$
We use the integrated Gamma driver 
$$
\partial_t \beta^i = \tilde{\Gamma}^i - \eta \beta^i + \beta^j \partial_j \beta^i 
$$

In [238]:
dalpha_dt = -S(2) * alpha * Khat 
for i in range(3):
    dalpha_dt += betau[i] * dalpha_dx_upw[i]

In [239]:
dbeta_dt = Gammatu - eta * betau 
for i in range(3):
    dbeta_dt += betau[i] * dbetau_dx_upw[i]

Hamiltonian constraints 
$$
\mathcal{H} = R + \tilde{A}_{ij} \tilde{A}^{ij} - \frac{2}{3} ( \hat{K} + 2 \Theta )^2 - 16 \pi \rho 
$$

In [240]:
# NB: some papers disagree on the signs here ... 
H_c = Rtrace - S(16) * sp.pi * rho + Rational(2,3) * ( Khat + 2*theta)**2 
for i in range(3):
    for j in range(3):
        H_c -= Atdd[i,j] * Atuu[i,j]

Momentum constraint
$$
M^i = \partial_j \tilde{A}^{ij} + \tilde{\Gamma}^i_{jk} \tilde{A}^{jk} - \frac{2}{3} \tilde{\gamma}^{ij} \partial_j \left( \hat{K} + 2\Theta \right) -\frac{2}{3} \tilde{A}^{ij} \partial_j \log{\chi} - 8 \pi \tilde{\gamma}^{ij} S_j 
$$
Where
$$
\partial_j \tilde{A}^{ij} = \partial_j \left( \tilde{\gamma}^{il} \tilde{\gamma}^{jk} \tilde{A}_{lk} \right) = \tilde{\gamma}^{il} \tilde{\gamma}^{jk} \partial_j \tilde{A}_{lk} - \tilde{A}^{i}_k (\tilde{\Gamma}_d)^k + \tilde{A}^{j}_l \partial_j \tilde{\gamma}^{li} = \tilde{\gamma}^{il} \tilde{\gamma}^{jk} \partial_j \tilde{A}_{lk} - \tilde{A}^{i}_k (\tilde{\Gamma}_d)^k -  \tilde{A}^{jl} \tilde{\gamma}^{ik} \partial_j \tilde{\gamma}_{lk} 
$$

In [241]:
dAtuu_dx = sp.Matrix([0]*3)
for i in range(3):
    for j in range(3):
        dAtuu_dx[i] -= Atud[i,j] * GammaDu[j]
        for k in range(3):
            for l in range(3):
                dAtuu_dx[i] += gtuu[i,l] * gtuu[j,k] * dAtdd_dx[j][l,k] - Atuu[j,l] * gtuu[i,k] * dgtdd_dx[j][l,k]


In [242]:
M_c = dAtuu_dx
for i in range(3):
    for j in range(3):
        M_c[i] += - Rational(2,3) * gtuu[i,j] * (dKhat_dx[j] + S(2) * dtheta_dx[j]) - Rational(2,3) * Atuu[i,j] * dchi_dx[j] / chi - S(8) * sp.pi * gtuu[i,j] * Sd[j]
        for k in range(3):
            M_c[i] += Gammatudd[i][j,k] * Atuu[j,k]

Now we write the matter source terms 
$$
\rho_{\rm ADM} = n^\mu n^\nu T_{\mu \nu}
$$
$$
S_i = -{\gamma^\mu}_i n^\nu T_{\mu\nu} = 
$$
$$
S_{ij} = {\gamma^\mu}_i  {\gamma^\nu}_j T_{\mu\nu}
$$

In [243]:
betad = gdd * betau 
g4dd = sp.Matrix([
    [-alpha**2 + betau.dot(betad), betad[0], betad[1], betad[2]],
    [betad[0], gdd[0,0], gdd[0,1], gdd[0,2]],
    [betad[1], gdd[1,0], gdd[1,1], gdd[1,2]],
    [betad[2], gdd[2,0], gdd[2,1], gdd[2,2]],
])
guu = chi * gtuu 
g4uu = sp.zeros(4,4)
g4uu[0,0] = -1/alpha**2
for mu in range(3):
    g4uu[0,mu+1] = g4uu[mu+1,0] = betau[mu]/alpha**2
    for nu in range(3):
        g4uu[mu+1,nu+1] = guu[mu,nu] - betau[mu] * betau[nu] / alpha**2 

n4u = sp.Matrix([1/alpha,-betau[0]/alpha,-betau[1]/alpha,-betau[2]/alpha])
n4d = sp.Matrix([-alpha,0,0,0])

II4 = sp.Matrix([
    [1,0,0,0],
    [0,1,0,0],
    [0,0,1,0],
    [0,0,0,1]
])
gamma_proj = II4 + n4u * n4d.T 

# Construct Tmunu 
rho0, P, eps = symbols("rho0 press eps", real=True)
zX, zY, zZ  = symbols("zvec[0] zvec[1] zvec[2]", real=True)
BX, BY, BZ  = symbols("Bvec[0] Bvec[1] Bvec[2]", real=True)
zvecu = sp.Matrix([zX,zY,zZ])
Bvecu = sp.Matrix([BX,BY,BZ])
Bvecd = gdd * Bvecu 
zvecd = gdd * zvecu 
B2 = Bvecu.dot(Bvecd)
# Basic derived qties 
h = 1 + eps + P/rho0
W = sp.sqrt(1+zvecu.dot(zvecd))
velu = zvecu / W 
veld = gdd * velu 
v2 = velu.dot(veld)

u0 = W / alpha 
u3u =  (alpha*velu-betau) * u0 
u4u = sp.Matrix([u0,*u3u])
u4d = g4dd * u4u 
u3d = sp.Matrix([u4d[1],u4d[2],u4d[3]])
# comoving B 
Bdotv = Bvecu.dot(veld)
bt_expl = Bvecu.dot(veld) * u0 
bi_expl = (Bvecu + alpha * bt_expl*u3u)/W
b2_expl = (B2 + alpha**2 * bt_expl**2)/W**2
b4u_expl = sp.Matrix([bt_expl,*bi_expl])
b4d = g4dd * b4u_expl

# Construct Tdownmunu 
Tdd = sp.zeros(4,4)

for mu in range(4):
    for nu in range(mu,4):
        Tdd[mu,nu] = Tdd[nu,mu] = (rho0*h + b2_expl) * u4d[mu] * u4d[nu] + (P + b2_expl/2) * g4dd[mu,nu] - b4d[mu] * b4d[nu]


rho_ADM = 0 
Sd_ADM = sp.zeros(3,1)
Sdd_ADM = sp.zeros(3,3)

for mu in range(4):
    for nu in range(4):
        rho_ADM +=  n4u[mu] * n4u[nu] * Tdd[mu,nu]

for i in range(3):
    for mu in range(4):
        for nu in range(4):
            Sd_ADM[i] += -gamma_proj[mu,i+1] * n4u[nu] * Tdd[mu,nu]

for i in range(3):
    for j in range(3):
        for mu in range(4):
            for nu in range(4):
                Sdd_ADM[i,j] += gamma_proj[mu,i+1] * gamma_proj[nu,j+1] * Tdd[mu,nu]

print(Sdd_ADM.is_symmetric())


trace_Sdd = sp.trace(guu*Sdd_ADM) 

True


In [244]:

# Energy and momentum consistency
test = 0
for eta in range(4):
    for mu in range(4):
        for nu in range(4):
            test += -gamma_proj[mu,eta] * n4u[nu] * Tdd[mu,nu] * n4u[eta]
print(test.simplify())

# Another test: g^munu S_munu == gamma^ij S_ij 
test = 0 
for eta in range(4):
    for zeta in range(4):
        for mu in range(4):
            for nu in range(4):
                test += gamma_proj[mu,eta] * gamma_proj[nu,zeta] * Tdd[mu,nu] * n4u[eta] * n4u[zeta]
print(sp.simplify(test))

0
0


In [245]:
# Write some code. ABI 
tens = ("double", [6])
scalar=("double",None)
vec = ("double",[3])
ABI = {
    "gtdd": tens,
    "Atdd": tens,
    "betau": vec,
    "alp": scalar,
    "chi": scalar,
    "Gammatu": vec,
    "theta": scalar,
    "Khat": scalar,
    "S": scalar,
    "rho": scalar,
    "Sij": tens,
    "Si": vec,
    "kappa1": scalar,
    "kappa2": scalar,
    "eta": scalar,
    # First derivatives
    "dgtdd_dx": ("double", [6*3]),
    "dgtdd_dx_upwind": ("double", [6*3]),
    "dAtdd_dx": ("double", [6*3]),
    "dAtdd_dx_upwind": ("double", [6*3]),
    "dbetau_dx": ("double", [3*3]),
    "dbetau_dx_upwind": ("double", [3*3]),
    "dGammatu_dx": ("double", [3*3]),
    "dGammatu_dx_upwind": ("double", [3*3]),
    "dKhat_dx": vec,
    "dKhat_dx_upwind": vec,
    "dchi_dx": vec,
    "dchi_dx_upwind": vec,
    "dalp_dx": vec,
    "dalp_dx_upwind": vec,
    "dtheta_dx": vec,
    "dtheta_dx_upwind": vec,
    # Second derivatives
    "ddgtdd_dx2": ("double", [6*6]),
    "ddAtdd_dx2": ("double", [6*6]),
    "ddbetau_dx2": ("double", [3*6]),
    "ddGammatu_dx2": ("double", [3*6]),
    "ddalp_dx2": ("double", [6]),
    "ddchi_dx2": ("double", [6]),
    "ddtheta_dx2": ("double", [6]),
    "ddKhat_dx2": ("double", [6]),
    # Matter 
    "zvec": vec,
    "Bvec": vec,
    "rho0": scalar,
    "press": scalar,
    "eps": scalar
}

In [246]:
flist = [] 

# RHS 
out_list = ["dchi", "dgtdd","dKhat","dGammatu","dtheta","dAtdd","dalp","dbetau"]
out_abi = {"dchi": scalar, "dgtdd": tens, "dKhat": scalar, "dGammatu": vec, "dtheta": scalar, "dAtdd":tens, "dalp": scalar, "dbetau": vec} 
exprs = [dchi_dt,dgtdd_dt,dKhat_dt,dGammatu_dt,dtheta_dt,dAtdd_dt,dalpha_dt,dbeta_dt]
flist.append(make_function(exprs,printer,"z4c_get_rhs",ABI,out_list,out_abi,cse_order='none',cse_optims=None))

# Constraints 
out_list = ["H","M"]
out_abi = {"H": scalar, "M": vec}
exprs=[H_c,M_c]
flist.append(make_function(exprs,printer,"z4c_get_constraints",ABI,out_list,out_abi))

# Constraints 
out_list = ["rho_adm","S_adm_trace", "Sd_adm","Sdd_adm"]
out_abi = {"rho_adm": scalar, "S_adm_trace": scalar, "Sd_adm": vec, "Sdd_adm": tens}
exprs=[rho_ADM,trace_Sdd,Sd_ADM,Sdd_ADM]
flist.append(make_function(exprs,printer,"z4c_get_matter_sources",ABI,out_list,out_abi))

printed_functions = '\n'+'\n'.join(flist)


In [247]:
file = '''
/****************************************************************************/
/*                       Z4C helpers, SymPy generated                       */
/****************************************************************************/
#ifndef GRACE_Z4C_SUBEXPR_HH
#define GRACE_Z4C_SUBEXPR_HH

#include <Kokkos_Core.hpp>
''' + printed_functions + '''
#endif 
'''

with open("z4c_subexpressions.hh","w") as ff:
    ff.write(file)